In [ ]:
# GATE 2 -- the notebook-to-notebook artifact hop, proven end to end before the
# full run generates ~5GB. The open question is file count, not total size:
# whether ~4,407 individual .npz files survive as a kernel output that a later
# kernel attaches via kernel_sources. This throwaway consumer kernel attaches
# the pilot's output and loads studies back through the real training loader.
#
# If this fails, the fallback is creating the dataset from inside the prep
# notebook via the Kaggle API with a token from Kaggle Secrets -- either way
# nothing gets pushed through the local connection.
import glob, os, shutil, sys, time

GIT_SHA = '9800bdc-wip'

src_candidates = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)
SRC = src_candidates[0]

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

from knee.dataset import PreppedStudyDataset
from knee.prep import load_study_npz
print('knee package imported successfully | GIT_SHA', GIT_SHA)

In [ ]:
# Find the prep kernels' outputs. Attached kernel outputs land under
# /kaggle/input/<kernel-slug>/, one directory per shard, so glob for every
# prepped/ directory rather than hard-coding a path or assuming a single shard.
prepped_dirs = sorted(glob.glob('/kaggle/input/**/prepped', recursive=True))
print(f'{len(prepped_dirs)} shard outputs attached:')
for d in prepped_dirs:
    print(f'  {d}: {len([f for f in os.listdir(d) if f.endswith(".npz")])} .npz')

# uid -> path, so a study prepped twice (overlapping shards) is caught rather
# than silently counted once
npz_paths = {}
duplicates = []
for d in prepped_dirs:
    for f in os.listdir(d):
        if not f.endswith('.npz'):
            continue
        uid = f[:-len('.npz')]
        if uid in npz_paths:
            duplicates.append(uid)
        npz_paths[uid] = os.path.join(d, f)

print(f'\n{len(npz_paths)} distinct studies across all shards, {len(duplicates)} duplicated')

manifests = sorted(glob.glob('/kaggle/input/**/prep_manifest_shard*.csv', recursive=True))
failed_csvs = sorted(glob.glob('/kaggle/input/**/prep_failed_shard*.csv', recursive=True))
print('manifests:', [os.path.basename(m) for m in manifests])

In [ ]:
# The count is the gate: every study the prep kernels wrote has to be readable
# here, and the union has to cover the whole corpus.
import numpy as np
import pandas as pd

manifest = pd.concat([pd.read_csv(p) for p in manifests], ignore_index=True)
failed = pd.concat([pd.read_csv(p) for p in failed_csvs], ignore_index=True)
expected = set(manifest['StudyInstanceUID'])
present = set(npz_paths)

print(f'manifest lists {len(expected)} studies, {len(present)} .npz files present')
print(f'studies that failed prep outright: {len(failed)}')
for _, row in failed.head(10).iterrows():
    print(' ', row['StudyInstanceUID'], row['error'])

missing = expected - present
extra = present - expected
print(f'missing: {len(missing)}, unexpected: {len(extra)}')
for uid in sorted(missing)[:10]:
    print('missing', uid)

print(f'\ngold studies prepped: {int(manifest["is_gold"].sum())} (expect 58)')

# The shards must partition the corpus, not merely cover it: verify each shard's
# UID set is exactly its own contiguous index range over the sorted corpus, so
# "0 duplicated" reflects a real partition rather than a lucky count.
import math
comp = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]
all_uids = sorted(
    d for d in os.listdir(f'{comp}/train_series')
    if os.path.isdir(os.path.join(f'{comp}/train_series', d))
)
per_shard = math.ceil(len(all_uids) / len(manifests))
partition_ok = True
for mpath in manifests:
    idx = int(os.path.basename(mpath).replace('prep_manifest_shard', '').replace('.csv', ''))
    got = set(pd.read_csv(mpath)['StudyInstanceUID'])
    want = set(all_uids[idx * per_shard:(idx + 1) * per_shard])
    ok = got == want
    partition_ok &= ok
    print(f'  shard {idx}: {len(got)} studies, range [{idx * per_shard}, '
          f'{min((idx + 1) * per_shard, len(all_uids))}) -- {"exact" if ok else "MISMATCH"}')
print(f'shards partition the corpus exactly: {partition_ok}')

# The gold holdout list Phase 3/4 depends on -- written by shard 0 only.
gold_csvs = glob.glob('/kaggle/input/**/gold_study_uids.csv', recursive=True)
print(f'\ngold_study_uids.csv found: {gold_csvs}')
gold_ok = bool(gold_csvs) and len(pd.read_csv(gold_csvs[0])) == 58
if gold_csvs:
    gold_listed = set(pd.read_csv(gold_csvs[0])['StudyInstanceUID'])
    print(f'  {len(gold_listed)} UIDs listed; all prepped: '
          f'{gold_listed <= present}; matches manifest is_gold: '
          f'{gold_listed == set(manifest.loc[manifest["is_gold"], "StudyInstanceUID"])}')
print(f'gold holdout list usable: {gold_ok}')
print('route mix over the full corpus:')
print(manifest['route'].value_counts())

In [ ]:
# Read every artifact back -- a hop that transfers the files but corrupts them
# is not a hop that passed.
t0 = time.time()
bad = []
for uid, path in npz_paths.items():
    try:
        series_slices, meta = load_study_npz(path)
        assert series_slices and meta['StudyInstanceUID'] == uid
    except Exception as exc:
        bad.append((uid, type(exc).__name__, str(exc)))
print(f'read {len(npz_paths)} artifacts in {time.time() - t0:.1f}s, {len(bad)} unreadable')
for row in bad[:10]:
    print(row)

In [ ]:
# And through the actual training loader, since that is what the full run is for.
# PreppedStudyDataset expects one root; symlink the shards into a single tree
# so training sees the corpus as one dataset rather than four
ROOT = '/kaggle/working/all_prepped'
os.makedirs(ROOT, exist_ok=True)
for uid, path in npz_paths.items():
    link = os.path.join(ROOT, f'{uid}.npz')
    if not os.path.exists(link):
        os.symlink(path, link)

loader = PreppedStudyDataset(sorted(present), ROOT, n_slices=24, max_series=4)
image, labels, uid = loader[0]
print(f'{uid}: volume {tuple(image.shape)}, dtype {image.dtype}, '
      f'range [{image.min():.3f}, {image.max():.3f}]')

per_study_ms = []
for idx in range(min(len(loader), 200)):
    t = time.time()
    loader[idx]
    per_study_ms.append((time.time() - t) * 1000)
per_study_ms = np.array(per_study_ms)
print(f'loader over the attached dataset: mean {per_study_ms.mean():.1f} ms/study, '
      f'p95 {np.percentile(per_study_ms, 95):.1f}')

passed = not bad and not missing and not duplicates and partition_ok and gold_ok
print(f'GATE 2: {"PASS" if passed else "FAIL -- fall back to Kaggle-API dataset creation"}')